In [1]:
import os 
import re 
import string 
import pandas as pd 
import numpy as np 
import mlflow 
import mlflow.sklearn 
import dagshub 
from sklearn.model_selection import train_test_split, GridSearchCV 
from sklearn.linear_model import LogisticRegression 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score 
from nltk.corpus import stopwords 
from nltk.stem import WordNetLemmatizer 
import nltk 
nltk.download('stopwords')
nltk.download('wordnet')

import warnings 
warnings.simplefilter("ignore", category = UserWarning)
warnings.filterwarnings("ignore")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
#MLFlow and Dagshub Setup 
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
dagshub.init(repo_owner=os.getenv("DAGSHUB_REPO_OWNER"), repo_name=os.getenv("DAGSHUB_REPO_NAME"), mlflow=True)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("LoR Hyperparameter Tuning")   #exp name inside mlflow

Accessing as MLayush-dubey

Initialized MLflow to track repo "MLayush-dubey/MLOps-IMDB-Sentiment-Analysis"

Repository MLayush-dubey/MLOps-IMDB-Sentiment-Analysis initialized!

<Experiment: artifact_location='mlflow-artifacts:/c4e995d90be64f69b1063b41ec8e504e', creation_time=1776164394202, experiment_id='2', last_update_time=1776164394202, lifecycle_stage='active', name='LoR Hyperparameter Tuning', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [3]:
#Preprocessing function 
def preprocess_text(text):
    lemmatizer = WordNetLemmatizer()
    stop_words = stopwords.words('english')

    text = str(text).lower()   #converts the text into a string(row-wise) and then lowercase
    text = re.sub(r'\d+', '', text)  #check for any digits and replace it with an empty string
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)   #check for punctuation and replace it with a space
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  #check for any URL and replace it with an empty string
    text = " ".join([word for word in text.split() if word not in stop_words])

    return text.strip()

The reason we convert it to a str--> to ensure that the text is in a format that can be processed by the preprocessing function.

Also: often NaN values are present in the dataset, so we need to convert them to `strings` from `float`

In [4]:
#Load and preprocess the data
df = pd.read_csv("C:\\Users\\PC\\Documents\\MLOps-IMDB-Sentiment-Analysis\\notebooks\\data.csv")

df["review"] = df["review"].astype(str).apply(preprocess_text)  #astype(str) converts the data column wise to a string(faster than str())

df = df[df["sentiment"].isin(["positive", "negative"])]
df["sentiment"] = df["sentiment"].map({"negative": 0, "positive": 1})

df.head()

,review,sentiment
0,yep lots shouting screaming cheering arguing c...,0
1,honest yall junior high school sitcom first ai...,1
2,liked solino much heart rending story italian ...,1
3,seeing forever hollywood would natural want se...,0
4,show great episodes one terrible episode hard ...,0


In [5]:
 #Vectorization and train-test-split 
vectorizer = TfidfVectorizer() 
X = vectorizer.fit_transform(df['review'])
Y = df['sentiment']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

print(f"Training shape: {X_train.shape}, Testing shape: {X_test.shape}")

Training shape: (400, 11935), Testing shape: (100, 11935)


> Tfidf counts number of unique words in a document; and harr unique word ka ek column bana deta hai hence high dimensionality

Note: Usually flow aisa rehta hai ki pehele x train x test meh divide krlete hai

fir x train pe vectorizer ko fit transform and x test pe transform

> Since humlog notebook experimentation pe hai, toh itna strict hone ki zarurat nahi hai

In [6]:
#hyperparameters
param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'max_iter': [50, 100, 200]
}

In [7]:
with mlflow.start_run():

    #GridSearchCV to find best hyperparameters 
    grid_search = GridSearchCV(LogisticRegression(), param_grid, cv = 5, scoring = "f1", n_jobs = -1)
    grid_search.fit(X_train, Y_train)

    #log each hyperparameter combination as a child run
    for params, mean_score, std_score in zip(grid_search.cv_results_["params"],grid_search.cv_results_["mean_test_score"], grid_search.cv_results_["std_test_score"]):
        with mlflow.start_run(run_name = f"LR with params: {params}", nested = True):

            model = LogisticRegression(**params)
            model.fit(X_train, Y_train)
            y_pred = model.predict(X_test)

            metrics = {
                "accuracy": accuracy_score(Y_test, y_pred),
                "precision": precision_score(Y_test, y_pred),
                "recall": recall_score(Y_test, y_pred),
                "f1": f1_score(Y_test, y_pred)
            }

            mlflow.log_metrics(metrics)
            mlflow.log_params(params)

            print(f"Params: {params} | Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f}")

    #log the best model in the parent run 
    best_params = grid_search.best_params_ 
    best_model = grid_search.best_estimator_ 
    best_f1 = grid_search.best_score_ 

    mlflow.log_params(best_params)
    mlflow.log_metric("best_f1_score", best_f1)
    mlflow.sklearn.log_model(best_model, "model")

    print(f"\nBest Params: {best_params} | Best F1: {best_f1:.4f}")

Params: {'C': 0.1, 'max_iter': 50, 'penalty': 'l1', 'solver': 'liblinear'} | Accuracy: 0.5200 | F1: 0.0000
🏃 View run LR with params: {'C': 0.1, 'max_iter': 50, 'penalty': 'l1', 'solver': 'liblinear'} at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/2/runs/d39f5ba031c943d99bfe1aa882cf2785
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/2
Params: {'C': 0.1, 'max_iter': 50, 'penalty': 'l2', 'solver': 'liblinear'} | Accuracy: 0.5400 | F1: 0.0800
🏃 View run LR with params: {'C': 0.1, 'max_iter': 50, 'penalty': 'l2', 'solver': 'liblinear'} at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/2/runs/9d61da78ee3b4c858ffc784da22c98b4
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/2
Params: {'C': 0.1, 'max_iter': 100, 'penalty': 'l1', 'solver': 'liblinear'} | Accuracy: 0.5200 | F1: 0.0000
🏃 View run LR 

2026/04/14 16:53:35 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



Best Params: {'C': 10, 'max_iter': 50, 'penalty': 'l2', 'solver': 'liblinear'} | Best F1: 0.7520
🏃 View run lyrical-croc-410 at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/2/runs/02e2ce2c12904881bbc59b25bccc1a77
🧪 View experiment at: https://dagshub.com/MLayush-dubey/MLOps-IMDB-Sentiment-Analysis.mlflow/#/experiments/2


Best params basically Cross Validation GridSearchCV meh se 5 folds ka best params. Toh best f1 is basically average of all 5 folds and for that best params are those that are mentioned.

Jo chize print kiye hai vo ek single test data ke hisaab se hai 